# nnU-Net segmentation quality versus OOD distance

Load **one fold/checkpoint** of the ZGT-trained 2D nnU-Net, extract pooled bottleneck
features, segment MAMA-MIA and INbreast, and correlate KNN distance to ZGT with Dice.
The full network stays loaded: the encoder-only loader from the feature notebooks
removes the decoder and therefore cannot be used for segmentation.

Features reuse `root_utils.nnunet_encoder_features.extract_nnunet_bottleneck`: trained
plan preprocessing, overlapping training-size tiles, deepest encoder spatial pooling,
and an average over tiles. Prediction uses nnU-Net's native preprocessing,
sliding-window inference, label conversion and restoration to **original image size**.
Ground-truth masks are used only to calculate metrics.

Run cells top to bottom after setting paths. Requires the existing nnUNet environment
plus `pytorch-ood`, `scikit-learn` (KNN), `scipy`, `matplotlib`, and Pillow.
This notebook supports single-channel 2D natural images with background=0 and lesion=1.

All experiment logic is defined and executed in this notebook; no separate pipeline script is required.


In [ ]:
MODEL_CODE_RELATIVE_PATH = "nnUNet"
from pathlib import Path
import json
import sys
import torch

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "pipelines_and_experiments":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "root_utils").is_dir():
    raise RuntimeError("Start Jupyter in FM_thesis or pipelines_and_experiments, or set PROJECT_ROOT explicitly.")

MODEL_CODE_DIR = PROJECT_ROOT / MODEL_CODE_RELATIVE_PATH
for path in [PROJECT_ROOT, MODEL_CODE_DIR]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))


DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

In [ ]:
# Update these paths for the machine on which inference will run.
DATASETS_ROOT = Path("/home/jovyan/thesis_project/datasets")
EVALUATION_ROOTS = {
    "mamamia": DATASETS_ROOT / "MamaMIA" / "extracted_slices",
    "inbreast": DATASETS_ROOT / "INbreast" / "INbreast Release 1.0" / "MassCases",
}
ZGT_DATA_ROOT = Path("/mnt/data/spathak")
# Prefer a CSV restricted to the ID training/reference split; mass rows are selected.
ZGT_METADATA_CSV = ZGT_DATA_ROOT / "CLaM-Annot-metadata.csv"

REEXTRACT_ID_FEATURES = True
ALLOW_LEGACY_ID_FEATURES = False
KNN_K = 50  # k-th nearest ID neighbor; lower this if using fewer than 50 ID images
KNN_NORMALIZE_FEATURES = True  # L2-normalize embeddings before computing distances
MAX_ID_CASES = None
MAX_EVALUATION_CASES_PER_DATASET = None  # e.g. 5 for a smoke run
SAVE_PRED_MASKS = False
IMAGE_NPZ_KEY = None  # None selects a role-specific key or the sole array
MASK_NPZ_KEY = None

NNUNET_MODEL_FOLDER = PROJECT_ROOT / "nnUNet" / "nnUNet_results" / "Dataset001_Mammography" / "nnUNetTrainer__nnUNetPlans__2d"
NNUNET_FOLD = 0
NNUNET_CHECKPOINT_NAME = "nnUNet_ZGT_500_epochs.pth"  # set to an existing checkpoint in fold_0
TILE_STEP_SIZE = 0.5
USE_MIRRORING = True  # native inference uses the axes allowed by training
OUTPUT_ROOT = PROJECT_ROOT / "results" / "pipelines_and_experiments" / "SEG_OOD_nnUNet"
ID_FEATURES_PATH = OUTPUT_ROOT / "zgt_id_features.pt"
# To reuse a raw tensor, set REEXTRACT_ID_FEATURES=False,
# ALLOW_LEGACY_ID_FEATURES=True and ID_FEATURES_PATH to
# PROJECT_ROOT / "feature_space_inspection" / "nnUNet_ZGT_allmasses".
# Check the SAME fold, checkpoint, plans and tile step were used for that tensor.

## Dataset loading

The pipeline definitions are included in notebook cells. These cells define helpers; the following execution cells run them.

In [ ]:
from __future__ import annotations
import csv
import hashlib
import json
import warnings
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F


def checkpoint_identity(path):
    """Include checkpoint path, size and modification time in cache metadata."""
    path = Path(path).expanduser().resolve()
    stat = path.stat()
    return {"path": str(path), "size": stat.st_size, "mtime_ns": stat.st_mtime_ns}


def collect_paired_cases(root, dataset, limit=None):
    """Read case folders containing image/img + mask in NPZ, NPY or image format.

    Point root at the desired split, e.g. extracted_slices or MassCases/Test.
    Relative image paths identify cases uniquely, including nested MRI slices.
    Ambiguous image/mask pairs are rejected instead of silently choosing one.
    """
    root = Path(root).expanduser().resolve()
    if not root.is_dir():
        raise FileNotFoundError(root)
    extensions = {".npz", ".npy", ".png", ".jpg", ".jpeg", ".tif", ".tiff"}
    images = sorted(p for p in root.rglob("*") if p.is_file()
                    and p.suffix.lower() in extensions
                    and p.stem.lower() in {"image", "img"}
                    and not any(part.startswith(".") for part in p.relative_to(root).parts))
    cases = []
    for image in images:
        siblings = [p for p in image.parent.iterdir() if p.is_file()
                    and p.suffix.lower() in extensions]
        masks = [p for p in siblings if p.stem.lower() == "mask"]
        image_candidates = [p for p in siblings if p.stem.lower() in {"image", "img"}]
        if len(masks) != 1 or len(image_candidates) != 1:
            raise ValueError(f"Expected exactly one image and mask in {image.parent}")
        cases.append({"dataset": dataset, "case_id": image.relative_to(root).as_posix(),
                      "image_path": str(image), "mask_path": str(masks[0])})
    return validate_cases(cases, limit)


def collect_zgt_cases(root, metadata_csv, limit=None):
    """Read mass images from the semicolon-delimited CLaM metadata CSV.

    Use a CSV containing the intended ID reference split. Multiple annotation
    rows for one image are merged into one case with a union ground-truth mask.
    """
    root = Path(root).expanduser().resolve()
    grouped = {}
    with Path(metadata_csv).open(newline="", encoding="utf-8-sig") as handle:
        reader = csv.DictReader(handle, delimiter=";")
        required = {"ImagePath", "ROIPath", "AbnormalityType"}
        if not required.issubset(reader.fieldnames or []):
            raise ValueError(f"ZGT CSV needs columns {sorted(required)}")
        for row in reader:
            if row["AbnormalityType"] != "Mass":
                continue
            image_name = row["ImagePath"].strip()
            mask_name = row["ROIPath"].strip()
            if not image_name or image_name.lower() in {"none", "nan"}:
                raise ValueError("Mass row has no ImagePath")
            case = grouped.setdefault(image_name, {
                "dataset": "zgt_id", "case_id": image_name,
                "image_path": str(root / image_name), "mask_paths": [],
            })
            if mask_name and mask_name.lower() not in {"none", "nan"}:
                mask_path = str(root / mask_name)
                if mask_path not in case["mask_paths"]:
                    case["mask_paths"].append(mask_path)
    return validate_cases([grouped[k] for k in sorted(grouped)], limit)


def validate_cases(cases, limit=None):
    if limit is not None and limit < 1:
        raise ValueError("limit must be positive or None")
    cases = list(cases)
    keys = [(c["dataset"], c["case_id"]) for c in cases]
    if len(keys) != len(set(keys)):
        raise ValueError("Duplicate (dataset, case_id) identifiers")
    cases = cases if limit is None else cases[:limit]
    if not cases:
        raise ValueError("No cases found; check the dataset root/split")
    for case in cases:
        paths = [case["image_path"]] + case.get("mask_paths", [case.get("mask_path")])
        for path in paths:
            if path is not None and not Path(path).is_file():
                raise FileNotFoundError(path)
    return cases


def load_array(path, role="image", npz_key=None):
    path = Path(path)
    if path.suffix.lower() == ".npy":
        return np.load(path, allow_pickle=False)
    if path.suffix.lower() == ".npz":
        with np.load(path, allow_pickle=False) as archive:
            if npz_key is not None:
                return archive[npz_key].copy()
            preferred = ("image", "img", "data", "arr_0") if role == "image" else ("mask", "seg", "label", "arr_0")
            if len(archive.files) == 1:
                return archive[archive.files[0]].copy()
            for key in preferred:
                if key in archive:
                    return archive[key].copy()
            raise ValueError(f"Select an NPZ key for {path}: {archive.files}")
    from PIL import Image
    with Image.open(path) as image:
        return np.asarray(image).copy()


def as_2d(array, role="image"):
    """Convert a grayscale/CHW/HWC slice to HW; reject volumes."""
    array = np.asarray(array)
    if array.ndim == 3:
        if array.shape[-1] in (1, 3, 4):
            array = (array[..., :3].mean(axis=-1).astype(array.dtype)
                     if role == "image" and array.shape[-1] > 1 else array[..., 0])
        elif array.shape[0] in (1, 3, 4):
            array = (array[:3].mean(axis=0).astype(array.dtype)
                     if role == "image" and array.shape[0] > 1 else array[0])
    if array.ndim != 2:
        raise ValueError(f"Expected a 2D {role} slice, got {array.shape}")
    if not np.isfinite(array).all():
        raise ValueError(f"Non-finite values in {role}")
    return array


def read_case(case, image_npz_key=None, mask_npz_key=None):
    image = as_2d(load_array(case["image_path"], "image", image_npz_key))
    paths = case.get("mask_paths", [case.get("mask_path")])
    mask = np.zeros(image.shape, dtype=bool)
    for path in paths:
        if path is None:
            continue
        component = as_2d(load_array(path, "mask", mask_npz_key), "mask")
        if component.shape != image.shape:
            raise ValueError(f"Image/mask shape mismatch for {case['case_id']}")
        mask |= component > 0
    return image, mask

## Feature extraction, segmentation evaluation, and OOD scoring

In [ ]:
def binary_metrics(prediction, target):
    prediction, target = np.asarray(prediction, dtype=bool), np.asarray(target, dtype=bool)
    if prediction.shape != target.shape or prediction.ndim != 2:
        raise ValueError(f"Expected matching HW masks: {prediction.shape}, {target.shape}")
    pred_area, gt_area = int(prediction.sum()), int(target.sum())
    intersection = int((prediction & target).sum())
    union = pred_area + gt_area - intersection
    return {"dice": 2 * intersection / (pred_area + gt_area) if pred_area + gt_area else 1.0,
            "iou": intersection / union if union else 1.0,
            "gt_area": gt_area, "pred_area": pred_area,
            "gt_fraction": gt_area / target.size,
            "evaluation_height": target.shape[0], "evaluation_width": target.shape[1]}


def extract_reference_features(cases, feature_fn, output_path, signature, **read_kwargs):
    vectors = []
    for index, case in enumerate(cases, 1):
        image, _ = read_case(case, **read_kwargs)
        vectors.append(feature_fn(image).detach().cpu().reshape(1, -1))
        if index % 10 == 0 or index == len(cases):
            print(f"ID features: {index}/{len(cases)}")
    features = torch.cat(vectors)
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({"features": features, "signature": signature, "cases": cases}, output_path)
    return features


def load_reference_features(path, signature, allow_legacy=False):
    payload = torch.load(path, map_location="cpu", weights_only=True)
    if torch.is_tensor(payload):
        if not allow_legacy:
            raise ValueError("Legacy ID tensor has no encoder/preprocessing metadata. Re-extract it "
                             "or set ALLOW_LEGACY_ID_FEATURES=True after checking its provenance.")
        warnings.warn("Using legacy ID features: verify the same encoder weights, fold and preprocessing")
        return payload.float()
    if payload["signature"] != signature:
        raise ValueError("ID feature encoder/preprocessing signature differs. Re-extract ID features.")
    return payload["features"].float()


def evaluate_cases(cases, feature_fn, predict_fn, output_dir, save_masks=False, **read_kwargs):
    """predict_fn(image) returns (binary prediction, evaluation HW)."""
    rows, vectors = [], []
    output_dir = Path(output_dir)
    for index, case in enumerate(cases, 1):
        image, original_gt = read_case(case, **read_kwargs)
        vector = feature_fn(image).detach().cpu().reshape(1, -1)
        prediction, evaluation_hw = predict_fn(image)
        if tuple(evaluation_hw) == original_gt.shape:
            gt = original_gt
        else:
            gt = F.interpolate(torch.from_numpy(original_gt.astype(np.float32))[None, None],
                               size=tuple(evaluation_hw), mode="nearest")[0, 0].numpy() > 0
        row = {**case, **binary_metrics(prediction, gt),
               "original_height": image.shape[0], "original_width": image.shape[1],
               "original_gt_area": int(original_gt.sum())}
        if save_masks:
            from PIL import Image
            # Hash the identity rather than flattening paths, which can collide.
            key = hashlib.sha256(f"{case['dataset']}:{case['case_id']}".encode()).hexdigest()[:20]
            path = output_dir / "pred_masks" / f"{key}.png"
            path.parent.mkdir(parents=True, exist_ok=True)
            Image.fromarray(np.asarray(prediction, dtype=np.uint8) * 255).save(path)
            row["prediction_path"] = str(path)
        rows.append(row)
        vectors.append(vector)
        if index % 10 == 0 or index == len(cases):
            print(f"Evaluation: {index}/{len(cases)}")
    if not vectors:
        raise ValueError("No evaluation cases")
    return rows, torch.cat(vectors)


def normalize_scores(raw_scores):
    scores = np.asarray(raw_scores, dtype=np.float64).reshape(-1)
    if not len(scores) or not np.isfinite(scores).all():
        raise ValueError("OOD scores must be nonempty and finite")
    low, high = float(scores.min()), float(scores.max())
    return ((scores - low) / (high - low) if high > low else np.zeros_like(scores),
            {"min": low, "max": high})


def attach_ood_scores(rows, id_features, evaluation_features, k=50, normalize=True):
    """K-th neighbor distance; defaults match SEG_OOD_medSAM's KNN detector."""
    from pytorch_ood.detector import KNN
    for name, matrix in [("ID", id_features), ("evaluation", evaluation_features)]:
        if matrix.ndim != 2 or not matrix.shape[0] or not torch.isfinite(matrix).all():
            raise ValueError(f"{name} features must be a nonempty finite (N, C) matrix")
    if id_features.shape[1] != evaluation_features.shape[1]:
        raise ValueError("ID/evaluation feature dimensions differ; check the encoder and checkpoint")
    if len(rows) != len(evaluation_features):
        raise ValueError("Number of feature vectors differs from number of metric rows")
    if not 1 <= k <= len(id_features):
        raise ValueError(f"KNN_K must be between 1 and the ID reference count ({len(id_features)})")
    detector = KNN(encoder=None, k=k, normalize=normalize)
    detector.fit_features(id_features.detach().cpu().float().clone(),
                          torch.zeros(len(id_features), dtype=torch.long))
    raw = torch.as_tensor(detector.predict_features(evaluation_features.detach().cpu().float().clone())).cpu().numpy()
    normalized, scaling = normalize_scores(raw)
    for row, raw_score, score in zip(rows, raw, normalized):
        row.update(ood_score_raw=float(raw_score), ood_score=float(score))
    return scaling

## Segmentation model definitions

In [ ]:
class NNUNetAdapter:
    """Keep the decoder loaded; use existing tiled encoder features + native inference."""
    def __init__(self, model_folder, device, fold=0, checkpoint_name="checkpoint_final.pth",
                 tile_step_size=0.5, use_mirroring=True):
        from nnunetv2.inference.predict_from_raw_data import nnUNetPredictor
        if not 0 < tile_step_size <= 1:
            raise ValueError("tile_step_size must be in (0, 1]")
        path = Path(model_folder) / f"fold_{fold}" / checkpoint_name
        if not path.is_file():
            raise FileNotFoundError(path)
        self.device, self.tile_step_size = torch.device(device), tile_step_size
        self.predictor = nnUNetPredictor(
            device=self.device, tile_step_size=tile_step_size, use_mirroring=use_mirroring,
            perform_everything_on_device=self.device.type == "cuda", verbose=False,
            verbose_preprocessing=False, allow_tqdm=False)
        self.predictor.initialize_from_trained_model_folder(
            str(model_folder), use_folds=(fold,), checkpoint_name=checkpoint_name)
        self.predictor.network.to(self.device).eval()
        if len(self.predictor.configuration_manager.patch_size) != 2:
            raise ValueError("This pipeline requires a 2D nnU-Net configuration")
        if len(self.predictor.dataset_json["channel_names"]) != 1:
            raise ValueError("This pipeline requires a single grayscale channel")
        labels = self.predictor.dataset_json["labels"]
        if set(labels.values()) != {0, 1}:
            raise ValueError("This pipeline expects background=0 and lesion=1")

    @torch.inference_mode()
    def features(self, image):
        from root_utils.nnunet_encoder_features import extract_nnunet_bottleneck
        p = self.predictor
        return extract_nnunet_bottleneck(
            image, p.network.encoder, p.plans_manager, p.configuration_manager,
            p.dataset_json, self.device, self.tile_step_size)

    @torch.inference_mode()
    def predict(self, image):
        from root_utils.nnunet_encoder_features import _as_nnunet_array
        segmentation = self.predictor.predict_single_npy_array(
            _as_nnunet_array(image), {"spacing": (999, 1, 1)},
            save_or_return_probabilities=False)
        segmentation = np.asarray(segmentation)
        if segmentation.ndim == 3 and segmentation.shape[0] == 1:
            segmentation = segmentation[0]
        # Native inference restores the original crop/shape and applies the
        # trained label manager; no extra sigmoid/0.5 threshold is appropriate.
        return segmentation == 1, image.shape

## Load the full nnU-Net

Keep feature extraction and segmentation on a single fold. Using an ensemble for predictions
while extracting features from only one fold changes the experiment's meaning.
Images retain their original intensity units and use the normalization/resampling in the
trained nnU-Net plans, matching the existing feature extractor. The `(999, 1, 1)` spacing
is the NaturalImage2DIO convention for these 2D images; it is not physical millimetre spacing.
No additional per-image min-max scaling or fixed 1024px resize is applied.

In [ ]:
adapter = NNUNetAdapter(
    NNUNET_MODEL_FOLDER, DEVICE, NNUNET_FOLD, NNUNET_CHECKPOINT_NAME,
    TILE_STEP_SIZE, USE_MIRRORING,
)
checkpoint_path = NNUNET_MODEL_FOLDER / f"fold_{NNUNET_FOLD}" / NNUNET_CHECKPOINT_NAME
feature_signature = {
    "encoder": "nnUNet deepest encoder spatial mean then tile mean",
    "checkpoint": checkpoint_identity(checkpoint_path),
    "fold": NNUNET_FOLD,
    "plans": adapter.predictor.plans_manager.plans,
    "configuration": adapter.predictor.configuration_manager.configuration,
    "dataset_json": adapter.predictor.dataset_json,
    "tile_step_size": TILE_STEP_SIZE,
    "spacing": [999, 1, 1],
}
segmentation_signature = {**feature_signature, "use_mirroring": USE_MIRRORING}
feature_fn = adapter.features
DICE_RESOLUTION = "original image resolution"
PLOT_TITLE = f"nnU-Net fold {NNUNET_FOLD}: Dice vs OOD distance"

## Select evaluation images

In [ ]:
evaluation_cases = []
for dataset_name, dataset_root in EVALUATION_ROOTS.items():
    cases = collect_paired_cases(dataset_root, dataset_name, MAX_EVALUATION_CASES_PER_DATASET)
    evaluation_cases.extend(cases)
    print(dataset_name, len(cases))

# No OOD ordering or filtering: all selected images are evaluated.
# For a different directory structure, replace evaluation_cases with records:
# {"dataset": "mamamia", "case_id": "unique_id", "image_path": "...", "mask_path": "..."}

## Extract or load the ID reference

In [ ]:
if REEXTRACT_ID_FEATURES:
    id_cases = collect_zgt_cases(ZGT_DATA_ROOT, ZGT_METADATA_CSV, MAX_ID_CASES)
    id_features = extract_reference_features(
        id_cases, feature_fn, ID_FEATURES_PATH, feature_signature,
    )
else:
    id_features = load_reference_features(
        ID_FEATURES_PATH, feature_signature, allow_legacy=ALLOW_LEGACY_ID_FEATURES,
    )
print("ID reference features:", tuple(id_features.shape))
if not 1 <= KNN_K <= len(id_features):
    raise ValueError(f"Set KNN_K between 1 and {len(id_features)} before evaluating images.")

## Extract evaluation features, run native inference, and score OOD distance

In [ ]:
rows, evaluation_features = evaluate_cases(
    evaluation_cases, feature_fn, adapter.predict, OUTPUT_ROOT,
    save_masks=SAVE_PRED_MASKS,
    image_npz_key=IMAGE_NPZ_KEY, mask_npz_key=MASK_NPZ_KEY,
)
score_scaling = attach_ood_scores(
    rows, id_features, evaluation_features, k=KNN_K, normalize=KNN_NORMALIZE_FEATURES,
)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
torch.save({"features": evaluation_features, "signature": feature_signature,
            "cases": evaluation_cases}, OUTPUT_ROOT / "evaluation_features.pt")
print("Joint OOD scaling:", score_scaling)

## Correlation analysis

In [ ]:
def save_analysis(rows, output_dir, title, config):
    """Save paired metrics, score-sorted ranking, per-dataset correlations and plot."""
    import matplotlib.pyplot as plt
    from scipy.stats import pearsonr, spearmanr

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    fields = sorted(set().union(*(row.keys() for row in rows)))
    for filename, records in [("per_image_metrics.csv", rows),
                              ("ranked_by_ood.csv", sorted(rows, key=lambda r: r["ood_score_raw"]))]:
        with (output_dir / filename).open("w", newline="") as handle:
            writer = csv.DictWriter(handle, fieldnames=fields)
            writer.writeheader()
            writer.writerows(records)
    fig, ax = plt.subplots(figsize=(8, 6))
    summary = {}
    for dataset in sorted({r["dataset"] for r in rows}):
        records = [r for r in rows if r["dataset"] == dataset]
        x = np.asarray([r["ood_score"] for r in records])
        y = np.asarray([r["dice"] for r in records])
        valid = np.isfinite(x) & np.isfinite(y)
        x, y = x[valid], y[valid]
        stats = {"n": len(x), "mean_dice": float(y.mean()) if len(y) else None,
                 "mean_ood_raw": float(np.mean([r["ood_score_raw"] for r in records])),
                 "pearson_r": None, "pearson_p": None, "spearman_r": None, "spearman_p": None}
        scatter = ax.scatter(x, y, alpha=0.6, label=dataset)
        if len(x) >= 2 and np.ptp(x) > 0 and np.ptp(y) > 0:
            r, p = pearsonr(x, y)
            rho, sp = spearmanr(x, y)
            stats.update(pearson_r=float(r), pearson_p=float(p) if np.isfinite(p) else None,
                         spearman_r=float(rho), spearman_p=float(sp) if np.isfinite(sp) else None)
            grid = np.linspace(x.min(), x.max(), 100)
            ax.plot(grid, np.polyval(np.polyfit(x, y, 1), grid),
                    color=scatter.get_facecolor()[0], label=f"{dataset}: r={r:.3f}")
        else:
            stats["correlation_note"] = "Requires at least two samples and nonconstant scores"
        summary[dataset] = stats
    ax.set(xlabel="OOD score (joint min-max normalization)", ylabel="Dice", title=title)
    ax.set_ylim(-0.02, 1.02)
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(output_dir / "dice_vs_ood.png", dpi=300)
    with (output_dir / "summary.json").open("w") as handle:
        json.dump({"datasets": summary, "config": config}, handle, indent=2, default=str, allow_nan=False)
    return fig, summary

In [ ]:
run_config = {
    "feature_signature": feature_signature,
    "segmentation_signature": segmentation_signature,
    "id_features_path": str(ID_FEATURES_PATH),
    "zgt_data_root": str(ZGT_DATA_ROOT), "zgt_metadata_csv": str(ZGT_METADATA_CSV),
    "reextracted_id_features": REEXTRACT_ID_FEATURES,
    "allow_legacy_id_features": ALLOW_LEGACY_ID_FEATURES,
    "id_reference_count": len(id_features),
    "evaluation_roots": {k: str(v) for k, v in EVALUATION_ROOTS.items()},
    "max_id_cases": MAX_ID_CASES,
    "max_evaluation_cases_per_dataset": MAX_EVALUATION_CASES_PER_DATASET,
    "image_npz_key": IMAGE_NPZ_KEY, "mask_npz_key": MASK_NPZ_KEY,
    "score_scaling": score_scaling,
    "dice_resolution": DICE_RESOLUTION,
    "empty_mask_dice": 1.0,
    "ood_detector": "pytorch_ood.detector.KNN",
    "knn_k": KNN_K, "knn_normalize_features": KNN_NORMALIZE_FEATURES,
}
figure, summary = save_analysis(rows, OUTPUT_ROOT, PLOT_TITLE, run_config)
print(json.dumps(summary, indent=2))
print("Results:", OUTPUT_ROOT)
figure

Each point is an image/slice with a paired OOD score and Dice. A negative correlation
supports an association between greater ID feature distance and worse segmentation.
The raw score is distance to the 50th nearest ID feature by default, after L2
normalizing both reference and evaluation vectors. Set `KNN_K` to change this.
The score is not an OOD probability. Joint min-max scaling uses both evaluation datasets;
retain raw scores when comparing different runs. Pearson and Spearman correlations are
computed **separately per dataset**, avoiding a pooled correlation driven by dataset differences.

Multiple slices from one MRI patient are not independent observations; the reported
p-values are descriptive image-level statistics. For patient-level inference, aggregate
by patient or use patient-level resampling. Foreground area/fraction is saved to help
inspect whether lesion size relates to either distance or Dice. Empty GT + empty prediction
has Dice 1; one empty mask has Dice 0.

Outputs: `per_image_metrics.csv`, `ranked_by_ood.csv` (ascending score, for inspection),
`summary.json`, `dice_vs_ood.png`, `evaluation_features.pt`, and optional `pred_masks/`.
Masks are stored at the evaluation resolution and referenced from the CSV.